# 02.02 — Operaciones con DataFrames en PySpark

## Objetivo

En este notebook practicaremos las operaciones más utilizadas para transformar y limpiar datos con la API de DataFrames de PySpark.

Trabajaremos principalmente con:

- `select()`
- `col()`
- `alias()`
- `filter()` y `where()`
- `withColumn()`
- `lit()`
- `when()` / `otherwise()`
- casteos de tipos
- valores nulos
- funciones de texto
- fechas
- ordenamiento
- eliminación de columnas y duplicados

El objetivo es desarrollar soltura con la API de DataFrames antes de avanzar hacia agregaciones, joins y funciones de ventana.

## 1. Crear la SparkSession

Comenzamos creando la sesión de Spark que utilizaremos durante el notebook.


In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    IntegerType,
    StringType,
    DoubleType
)

spark = (
    SparkSession.builder
    .appName("02_02_dataframe_operations")
    .getOrCreate()
)

Comprobamos el entorno:

In [2]:
print(f"Spark version: {spark.version}")
print(f"Master: {spark.sparkContext.master}")
print(f"Default parallelism: {spark.sparkContext.defaultParallelism}")

Spark version: 4.1.3
Master: local[*]
Default parallelism: 12


## 2. Dataset de trabajo

Para practicar las operaciones utilizaremos un pequeño dataset de clientes.

Incluiremos deliberadamente:

- diferentes edades;
- distintas ciudades;
- ingresos;
- fechas de registro;
- valores nulos;
- categorías;
- registros con características distintas.

Esto nos permitirá practicar transformaciones similares a las que encontraríamos en un proceso real de limpieza y preparación de datos.

###  2.1 Datos

In [3]:
data = [
    (1, "Ana", "CDMX",       28, 18500.0, "2026-01-15", "Premium",  "ana@email.com"),
    (2, "Luis", "Jalisco",    35, 24000.0, "2025-11-03", "Básico",   "luis@email.com"),
    (3, "Marta", "Nuevo León",42, 31500.0, "2024-07-21", "Premium",  None),
    (4, "Carlos", "CDMX",     31, None,    "2026-02-10", "Básico",   "carlos@email.com"),
    (5, "Sofía", "Puebla",    24, 16200.0, "2025-08-18", "Básico",   "sofia@email.com"),
    (6, "Jorge", "Jalisco",   53, 42000.0, "2023-05-06", "Premium",  "jorge@email.com"),
    (7, "Elena", "CDMX",      47, 37000.0, "2024-12-11", "Premium",  None),
    (8, "Diego", "Querétaro", 29, 21500.0, "2026-03-02", None,       "diego@email.com"),
    (9, "Laura", "Puebla",    38, 27800.0, "2025-02-14", "Premium",  "laura@email.com"),
    (10, "Pedro", "CDMX",     61, 46500.0, "2022-09-30", "Premium",  "pedro@email.com"),
    (11, "Andrea", "Jalisco",  33, 23200.0, None,         "Básico",   "andrea@email.com"),
    (12, "Miguel", None,       45, 34000.0, "2024-04-19", "Básico",   "miguel@email.com")
]

### 2.2 Schema explícito

Definiremos el schema manualmente para mantener control sobre los tipos de datos.

In [4]:
schema = StructType([
    StructField("id_cliente", IntegerType(), False),
    StructField("nombre", StringType(), False),
    StructField("estado", StringType(), True),
    StructField("edad", IntegerType(), True),
    StructField("ingreso_mensual", DoubleType(), True),
    StructField("fecha_registro", StringType(), True),
    StructField("tipo_cliente", StringType(), True),
    StructField("email", StringType(), True)
])

Creamos el DataFrame:

In [5]:
df_clientes = spark.createDataFrame(
    data,
    schema=schema
)

df_clientes.show(truncate=False)

+----------+------+----------+----+---------------+--------------+------------+----------------+
|id_cliente|nombre|estado    |edad|ingreso_mensual|fecha_registro|tipo_cliente|email           |
+----------+------+----------+----+---------------+--------------+------------+----------------+
|1         |Ana   |CDMX      |28  |18500.0        |2026-01-15    |Premium     |ana@email.com   |
|2         |Luis  |Jalisco   |35  |24000.0        |2025-11-03    |Básico      |luis@email.com  |
|3         |Marta |Nuevo León|42  |31500.0        |2024-07-21    |Premium     |NULL            |
|4         |Carlos|CDMX      |31  |NULL           |2026-02-10    |Básico      |carlos@email.com|
|5         |Sofía |Puebla    |24  |16200.0        |2025-08-18    |Básico      |sofia@email.com |
|6         |Jorge |Jalisco   |53  |42000.0        |2023-05-06    |Premium     |jorge@email.com |
|7         |Elena |CDMX      |47  |37000.0        |2024-12-11    |Premium     |NULL            |
|8         |Diego |Querétaro |

### 2.3 Inspección inicial

Antes de transformar datos conviene revisar su estructura.

In [6]:
df_clientes.printSchema()

root
 |-- id_cliente: integer (nullable = false)
 |-- nombre: string (nullable = false)
 |-- estado: string (nullable = true)
 |-- edad: integer (nullable = true)
 |-- ingreso_mensual: double (nullable = true)
 |-- fecha_registro: string (nullable = true)
 |-- tipo_cliente: string (nullable = true)
 |-- email: string (nullable = true)



También podemos consultar:

In [7]:
print("Columnas:")
print(df_clientes.columns)

print("\nNúmero de registros:")
print(df_clientes.count())

Columnas:
['id_cliente', 'nombre', 'estado', 'edad', 'ingreso_mensual', 'fecha_registro', 'tipo_cliente', 'email']

Número de registros:
12


## 3. Selección de columnas con `select()`

`select()` permite elegir qué columnas queremos conservar en un DataFrame.

Por ejemplo:

In [8]:
df_clientes.select(
    "id_cliente",
    "nombre",
    "estado"
).show()

+----------+------+----------+
|id_cliente|nombre|    estado|
+----------+------+----------+
|         1|   Ana|      CDMX|
|         2|  Luis|   Jalisco|
|         3| Marta|Nuevo León|
|         4|Carlos|      CDMX|
|         5| Sofía|    Puebla|
|         6| Jorge|   Jalisco|
|         7| Elena|      CDMX|
|         8| Diego| Querétaro|
|         9| Laura|    Puebla|
|        10| Pedro|      CDMX|
|        11|Andrea|   Jalisco|
|        12|Miguel|      NULL|
+----------+------+----------+



El DataFrame original no se modifica.

### 3.1 Usar `F.col()`

Otra forma de seleccionar columnas es mediante `F.col()`.

In [9]:
df_clientes.select(
    F.col("id_cliente"),
    F.col("nombre"),
    F.col("ingreso_mensual")
).show()

+----------+------+---------------+
|id_cliente|nombre|ingreso_mensual|
+----------+------+---------------+
|         1|   Ana|        18500.0|
|         2|  Luis|        24000.0|
|         3| Marta|        31500.0|
|         4|Carlos|           NULL|
|         5| Sofía|        16200.0|
|         6| Jorge|        42000.0|
|         7| Elena|        37000.0|
|         8| Diego|        21500.0|
|         9| Laura|        27800.0|
|        10| Pedro|        46500.0|
|        11|Andrea|        23200.0|
|        12|Miguel|        34000.0|
+----------+------+---------------+



Aunque para selecciones simples utilizar strings es suficiente, `F.col()` será especialmente útil cuando construyamos expresiones.

Por ejemplo:

In [10]:
df_clientes.select(
    F.col("nombre"),
    F.col("ingreso_mensual"),
    (F.col("ingreso_mensual") * 12).alias("ingreso_anual")
).show()

+------+---------------+-------------+
|nombre|ingreso_mensual|ingreso_anual|
+------+---------------+-------------+
|   Ana|        18500.0|     222000.0|
|  Luis|        24000.0|     288000.0|
| Marta|        31500.0|     378000.0|
|Carlos|           NULL|         NULL|
| Sofía|        16200.0|     194400.0|
| Jorge|        42000.0|     504000.0|
| Elena|        37000.0|     444000.0|
| Diego|        21500.0|     258000.0|
| Laura|        27800.0|     333600.0|
| Pedro|        46500.0|     558000.0|
|Andrea|        23200.0|     278400.0|
|Miguel|        34000.0|     408000.0|
+------+---------------+-------------+



Aquí estamos creando una nueva expresión directamente dentro de `select()`.

## 4. Renombrar expresiones con `alias()`

`alias()` permite asignar un nombre a una columna o expresión.

Por ejemplo:

In [11]:
df_clientes.select(
    F.col("nombre"),
    F.col("ingreso_mensual").alias("ingreso")
).show()

+------+-------+
|nombre|ingreso|
+------+-------+
|   Ana|18500.0|
|  Luis|24000.0|
| Marta|31500.0|
|Carlos|   NULL|
| Sofía|16200.0|
| Jorge|42000.0|
| Elena|37000.0|
| Diego|21500.0|
| Laura|27800.0|
| Pedro|46500.0|
|Andrea|23200.0|
|Miguel|34000.0|
+------+-------+



También resulta útil cuando construimos columnas calculadas:

In [12]:
df_clientes.select(
    F.col("nombre"),
    F.col("ingreso_mensual"),
    (F.col("ingreso_mensual") * 12).alias("ingreso_anual")
).show()

+------+---------------+-------------+
|nombre|ingreso_mensual|ingreso_anual|
+------+---------------+-------------+
|   Ana|        18500.0|     222000.0|
|  Luis|        24000.0|     288000.0|
| Marta|        31500.0|     378000.0|
|Carlos|           NULL|         NULL|
| Sofía|        16200.0|     194400.0|
| Jorge|        42000.0|     504000.0|
| Elena|        37000.0|     444000.0|
| Diego|        21500.0|     258000.0|
| Laura|        27800.0|     333600.0|
| Pedro|        46500.0|     558000.0|
|Andrea|        23200.0|     278400.0|
|Miguel|        34000.0|     408000.0|
+------+---------------+-------------+



Esto evita nombres poco legibles generados automáticamente por Spark.

## 5. Filtrar registros

Para seleccionar únicamente los registros que cumplen una condición podemos utilizar:

```python
filter()
```

Por ejemplo, clientes mayores de 35 años:

In [13]:
df_clientes.filter(
    F.col("edad") > 35
).show()

+----------+------+----------+----+---------------+--------------+------------+----------------+
|id_cliente|nombre|    estado|edad|ingreso_mensual|fecha_registro|tipo_cliente|           email|
+----------+------+----------+----+---------------+--------------+------------+----------------+
|         3| Marta|Nuevo León|  42|        31500.0|    2024-07-21|     Premium|            NULL|
|         6| Jorge|   Jalisco|  53|        42000.0|    2023-05-06|     Premium| jorge@email.com|
|         7| Elena|      CDMX|  47|        37000.0|    2024-12-11|     Premium|            NULL|
|         9| Laura|    Puebla|  38|        27800.0|    2025-02-14|     Premium| laura@email.com|
|        10| Pedro|      CDMX|  61|        46500.0|    2022-09-30|     Premium| pedro@email.com|
|        12|Miguel|      NULL|  45|        34000.0|    2024-04-19|      Básico|miguel@email.com|
+----------+------+----------+----+---------------+--------------+------------+----------------+



También podemos utilizar:

```python
where()
```

In [14]:
df_clientes.where(
    F.col("edad") > 35
).show()

+----------+------+----------+----+---------------+--------------+------------+----------------+
|id_cliente|nombre|    estado|edad|ingreso_mensual|fecha_registro|tipo_cliente|           email|
+----------+------+----------+----+---------------+--------------+------------+----------------+
|         3| Marta|Nuevo León|  42|        31500.0|    2024-07-21|     Premium|            NULL|
|         6| Jorge|   Jalisco|  53|        42000.0|    2023-05-06|     Premium| jorge@email.com|
|         7| Elena|      CDMX|  47|        37000.0|    2024-12-11|     Premium|            NULL|
|         9| Laura|    Puebla|  38|        27800.0|    2025-02-14|     Premium| laura@email.com|
|        10| Pedro|      CDMX|  61|        46500.0|    2022-09-30|     Premium| pedro@email.com|
|        12|Miguel|      NULL|  45|        34000.0|    2024-04-19|      Básico|miguel@email.com|
+----------+------+----------+----+---------------+--------------+------------+----------------+



En este contexto, ambos métodos pueden utilizarse de forma equivalente.

### 5.1 Múltiples condiciones

Podemos combinar condiciones utilizando:

```text
&  → AND
|  → OR
~  → NOT
```

#### Condición AND
Por ejemplo, clientes mayores de 30 años con ingresos superiores a 25,000:

In [15]:
df_clientes.filter(
    (F.col("edad") > 30) &
    (F.col("ingreso_mensual") > 25000)
).show()

+----------+------+----------+----+---------------+--------------+------------+----------------+
|id_cliente|nombre|    estado|edad|ingreso_mensual|fecha_registro|tipo_cliente|           email|
+----------+------+----------+----+---------------+--------------+------------+----------------+
|         3| Marta|Nuevo León|  42|        31500.0|    2024-07-21|     Premium|            NULL|
|         6| Jorge|   Jalisco|  53|        42000.0|    2023-05-06|     Premium| jorge@email.com|
|         7| Elena|      CDMX|  47|        37000.0|    2024-12-11|     Premium|            NULL|
|         9| Laura|    Puebla|  38|        27800.0|    2025-02-14|     Premium| laura@email.com|
|        10| Pedro|      CDMX|  61|        46500.0|    2022-09-30|     Premium| pedro@email.com|
|        12|Miguel|      NULL|  45|        34000.0|    2024-04-19|      Básico|miguel@email.com|
+----------+------+----------+----+---------------+--------------+------------+----------------+



Es importante colocar cada condición entre paréntesis.

#### Condición OR

Clientes que vivan en CDMX o Jalisco:

In [16]:
df_clientes.filter(
    (F.col("estado") == "CDMX") |
    (F.col("estado") == "Jalisco")
).show()

+----------+------+-------+----+---------------+--------------+------------+----------------+
|id_cliente|nombre| estado|edad|ingreso_mensual|fecha_registro|tipo_cliente|           email|
+----------+------+-------+----+---------------+--------------+------------+----------------+
|         1|   Ana|   CDMX|  28|        18500.0|    2026-01-15|     Premium|   ana@email.com|
|         2|  Luis|Jalisco|  35|        24000.0|    2025-11-03|      Básico|  luis@email.com|
|         4|Carlos|   CDMX|  31|           NULL|    2026-02-10|      Básico|carlos@email.com|
|         6| Jorge|Jalisco|  53|        42000.0|    2023-05-06|     Premium| jorge@email.com|
|         7| Elena|   CDMX|  47|        37000.0|    2024-12-11|     Premium|            NULL|
|        10| Pedro|   CDMX|  61|        46500.0|    2022-09-30|     Premium| pedro@email.com|
|        11|Andrea|Jalisco|  33|        23200.0|          NULL|      Básico|andrea@email.com|
+----------+------+-------+----+---------------+------------

También podemos utilizar:

In [17]:
df_clientes.filter(
    F.col("estado").isin("CDMX", "Jalisco")
).show()

+----------+------+-------+----+---------------+--------------+------------+----------------+
|id_cliente|nombre| estado|edad|ingreso_mensual|fecha_registro|tipo_cliente|           email|
+----------+------+-------+----+---------------+--------------+------------+----------------+
|         1|   Ana|   CDMX|  28|        18500.0|    2026-01-15|     Premium|   ana@email.com|
|         2|  Luis|Jalisco|  35|        24000.0|    2025-11-03|      Básico|  luis@email.com|
|         4|Carlos|   CDMX|  31|           NULL|    2026-02-10|      Básico|carlos@email.com|
|         6| Jorge|Jalisco|  53|        42000.0|    2023-05-06|     Premium| jorge@email.com|
|         7| Elena|   CDMX|  47|        37000.0|    2024-12-11|     Premium|            NULL|
|        10| Pedro|   CDMX|  61|        46500.0|    2022-09-30|     Premium| pedro@email.com|
|        11|Andrea|Jalisco|  33|        23200.0|          NULL|      Básico|andrea@email.com|
+----------+------+-------+----+---------------+------------

La segunda opción suele ser más cómoda cuando tenemos varias categorías.

## 6. Crear columnas con `withColumn()`

`withColumn()` permite crear una nueva columna o reemplazar una existente.

Crearemos el ingreso anual:

In [18]:
df_ingresos = (
    df_clientes
    .withColumn(
        "ingreso_anual",
        F.col("ingreso_mensual") * 12
    )
)

df_ingresos.select(
    "nombre",
    "ingreso_mensual",
    "ingreso_anual"
).show()

+------+---------------+-------------+
|nombre|ingreso_mensual|ingreso_anual|
+------+---------------+-------------+
|   Ana|        18500.0|     222000.0|
|  Luis|        24000.0|     288000.0|
| Marta|        31500.0|     378000.0|
|Carlos|           NULL|         NULL|
| Sofía|        16200.0|     194400.0|
| Jorge|        42000.0|     504000.0|
| Elena|        37000.0|     444000.0|
| Diego|        21500.0|     258000.0|
| Laura|        27800.0|     333600.0|
| Pedro|        46500.0|     558000.0|
|Andrea|        23200.0|     278400.0|
|Miguel|        34000.0|     408000.0|
+------+---------------+-------------+



Como vimos en el notebook anterior, esta operación devuelve un nuevo DataFrame.

### 6.1 Crear varias columnas

Podemos encadenar varias transformaciones:

In [19]:
df_ingresos = (
    df_clientes
    .withColumn(
        "ingreso_anual",
        F.col("ingreso_mensual") * 12
    )
    .withColumn(
        "ingreso_diario_aprox",
        F.col("ingreso_mensual") / 30
    )
)

df_ingresos.select(
    "nombre",
    "ingreso_mensual",
    "ingreso_anual",
    "ingreso_diario_aprox"
).show()

+------+---------------+-------------+--------------------+
|nombre|ingreso_mensual|ingreso_anual|ingreso_diario_aprox|
+------+---------------+-------------+--------------------+
|   Ana|        18500.0|     222000.0|   616.6666666666666|
|  Luis|        24000.0|     288000.0|               800.0|
| Marta|        31500.0|     378000.0|              1050.0|
|Carlos|           NULL|         NULL|                NULL|
| Sofía|        16200.0|     194400.0|               540.0|
| Jorge|        42000.0|     504000.0|              1400.0|
| Elena|        37000.0|     444000.0|  1233.3333333333333|
| Diego|        21500.0|     258000.0|   716.6666666666666|
| Laura|        27800.0|     333600.0|   926.6666666666666|
| Pedro|        46500.0|     558000.0|              1550.0|
|Andrea|        23200.0|     278400.0|   773.3333333333334|
|Miguel|        34000.0|     408000.0|  1133.3333333333333|
+------+---------------+-------------+--------------------+



## 7. Agregar valores constantes con `lit()`

`F.lit()` permite crear una columna con un valor constante.

Por ejemplo:

In [20]:
df_con_pais = (
    df_clientes
    .withColumn(
        "pais",
        F.lit("México")
    )
)

df_con_pais.select(
    "nombre",
    "estado",
    "pais"
).show()

+------+----------+------+
|nombre|    estado|  pais|
+------+----------+------+
|   Ana|      CDMX|México|
|  Luis|   Jalisco|México|
| Marta|Nuevo León|México|
|Carlos|      CDMX|México|
| Sofía|    Puebla|México|
| Jorge|   Jalisco|México|
| Elena|      CDMX|México|
| Diego| Querétaro|México|
| Laura|    Puebla|México|
| Pedro|      CDMX|México|
|Andrea|   Jalisco|México|
|Miguel|      NULL|México|
+------+----------+------+



Esto puede resultar útil cuando necesitamos agregar información fija a todos los registros.

## 8. Crear categorías con `when()` y `otherwise()`

Una operación muy común en preparación de datos consiste en crear categorías a partir de reglas.

Spark permite hacerlo mediante:

```python
F.when()
```

Vamos a crear grupos de edad.

In [21]:
df_segmentado = (
    df_clientes
    .withColumn(
        "grupo_edad",
        F.when(F.col("edad") < 30, "Menor de 30")
         .when(F.col("edad") < 40, "30-39")
         .when(F.col("edad") < 50, "40-49")
         .otherwise("50 o más")
    )
)

df_segmentado.select(
    "nombre",
    "edad",
    "grupo_edad"
).show()

+------+----+-----------+
|nombre|edad| grupo_edad|
+------+----+-----------+
|   Ana|  28|Menor de 30|
|  Luis|  35|      30-39|
| Marta|  42|      40-49|
|Carlos|  31|      30-39|
| Sofía|  24|Menor de 30|
| Jorge|  53|   50 o más|
| Elena|  47|      40-49|
| Diego|  29|Menor de 30|
| Laura|  38|      30-39|
| Pedro|  61|   50 o más|
|Andrea|  33|      30-39|
|Miguel|  45|      40-49|
+------+----+-----------+



Conceptualmente:

```text
edad < 30
    ↓
Menor de 30

30 <= edad < 40
    ↓
30-39

40 <= edad < 50
    ↓
40-49

edad >= 50
    ↓
50 o más
```

### 8.1 Crear una clasificación por ingreso

Podemos aplicar el mismo patrón a otra variable.

In [22]:
df_segmentado = (
    df_segmentado
    .withColumn(
        "nivel_ingreso",
        F.when(
            F.col("ingreso_mensual").isNull(),
            "Sin información"
        )
        .when(
            F.col("ingreso_mensual") < 20000,
            "Bajo"
        )
        .when(
            F.col("ingreso_mensual") < 30000,
            "Medio"
        )
        .otherwise("Alto")
    )
)

df_segmentado.select(
    "nombre",
    "ingreso_mensual",
    "nivel_ingreso"
).show()

+------+---------------+---------------+
|nombre|ingreso_mensual|  nivel_ingreso|
+------+---------------+---------------+
|   Ana|        18500.0|           Bajo|
|  Luis|        24000.0|          Medio|
| Marta|        31500.0|           Alto|
|Carlos|           NULL|Sin información|
| Sofía|        16200.0|           Bajo|
| Jorge|        42000.0|           Alto|
| Elena|        37000.0|           Alto|
| Diego|        21500.0|          Medio|
| Laura|        27800.0|          Medio|
| Pedro|        46500.0|           Alto|
|Andrea|        23200.0|          Medio|
|Miguel|        34000.0|           Alto|
+------+---------------+---------------+



Aquí introducimos también nuestra primera validación explícita de valores nulos.

## 9. Primera transformación integrada

Antes de avanzar hacia casteos y tratamiento de nulos, combinaremos varias operaciones en un pequeño pipeline.

In [23]:
df_transformado = (
    df_clientes
    .filter(
        F.col("edad") >= 30
    )
    .withColumn(
        "ingreso_anual",
        F.col("ingreso_mensual") * 12
    )
    .withColumn(
        "grupo_edad",
        F.when(F.col("edad") < 40, "30-39")
         .when(F.col("edad") < 50, "40-49")
         .otherwise("50 o más")
    )
    .withColumn(
        "pais",
        F.lit("México")
    )
    .select(
        "id_cliente",
        "nombre",
        "estado",
        "edad",
        "grupo_edad",
        "ingreso_mensual",
        "ingreso_anual",
        "pais"
    )
)

df_transformado.show(truncate=False)

+----------+------+----------+----+----------+---------------+-------------+------+
|id_cliente|nombre|estado    |edad|grupo_edad|ingreso_mensual|ingreso_anual|pais  |
+----------+------+----------+----+----------+---------------+-------------+------+
|2         |Luis  |Jalisco   |35  |30-39     |24000.0        |288000.0     |México|
|3         |Marta |Nuevo León|42  |40-49     |31500.0        |378000.0     |México|
|4         |Carlos|CDMX      |31  |30-39     |NULL           |NULL         |México|
|6         |Jorge |Jalisco   |53  |50 o más  |42000.0        |504000.0     |México|
|7         |Elena |CDMX      |47  |40-49     |37000.0        |444000.0     |México|
|9         |Laura |Puebla    |38  |30-39     |27800.0        |333600.0     |México|
|10        |Pedro |CDMX      |61  |50 o más  |46500.0        |558000.0     |México|
|11        |Andrea|Jalisco   |33  |30-39     |23200.0        |278400.0     |México|
|12        |Miguel|NULL      |45  |40-49     |34000.0        |408000.0     |

En una sola cadena hemos aplicado:

```text
filter()
    ↓
withColumn()
    ↓
withColumn()
    ↓
withColumn()
    ↓
select()
```

Este estilo de trabajo será cada vez más común a medida que construyamos pipelines de transformación más grandes.

## 10. Conversión de tipos con `cast()`

En datasets reales es común encontrar columnas almacenadas con tipos que no son los adecuados para el análisis.

PySpark permite convertir columnas utilizando:

```python
cast()
```

Por ejemplo, podemos convertir la edad a tipo `double`:

In [24]:
df_clientes.select(
    "nombre",
    F.col("edad"),
    F.col("edad").cast("double").alias("edad_double")
).show()

+------+----+-----------+
|nombre|edad|edad_double|
+------+----+-----------+
|   Ana|  28|       28.0|
|  Luis|  35|       35.0|
| Marta|  42|       42.0|
|Carlos|  31|       31.0|
| Sofía|  24|       24.0|
| Jorge|  53|       53.0|
| Elena|  47|       47.0|
| Diego|  29|       29.0|
| Laura|  38|       38.0|
| Pedro|  61|       61.0|
|Andrea|  33|       33.0|
|Miguel|  45|       45.0|
+------+----+-----------+



También podemos utilizar tipos de Spark directamente:

In [25]:
df_clientes.select(
    F.col("id_cliente").cast("string").alias("id_cliente_string"),
    "nombre"
).show()

+-----------------+------+
|id_cliente_string|nombre|
+-----------------+------+
|                1|   Ana|
|                2|  Luis|
|                3| Marta|
|                4|Carlos|
|                5| Sofía|
|                6| Jorge|
|                7| Elena|
|                8| Diego|
|                9| Laura|
|               10| Pedro|
|               11|Andrea|
|               12|Miguel|
+-----------------+------+



### 10.1 Revisar los tipos después de una conversión

Podemos crear un nuevo DataFrame con columnas modificadas:

In [26]:
df_cast = (
    df_clientes
    .withColumn(
        "edad_double",
        F.col("edad").cast("double")
    )
    .withColumn(
        "id_cliente_string",
        F.col("id_cliente").cast("string")
    )
)

df_cast.printSchema()

root
 |-- id_cliente: integer (nullable = false)
 |-- nombre: string (nullable = false)
 |-- estado: string (nullable = true)
 |-- edad: integer (nullable = true)
 |-- ingreso_mensual: double (nullable = true)
 |-- fecha_registro: string (nullable = true)
 |-- tipo_cliente: string (nullable = true)
 |-- email: string (nullable = true)
 |-- edad_double: double (nullable = true)
 |-- id_cliente_string: string (nullable = false)



Es importante recordar que `cast()` no modifica el DataFrame original.

## 11. Tratamiento de valores nulos

Los valores faltantes son uno de los problemas más comunes durante la preparación de datos.

En nuestro dataset existen valores nulos en columnas como:

- `estado`
- `ingreso_mensual`
- `fecha_registro`
- `tipo_cliente`
- `email`

Primero identificaremos dónde se encuentran.

### 11.1 Detectar valores nulos

Podemos utilizar:

```python
isNull()
```

Por ejemplo:

In [27]:
df_clientes.filter(
    F.col("ingreso_mensual").isNull()
).show()

+----------+------+------+----+---------------+--------------+------------+----------------+
|id_cliente|nombre|estado|edad|ingreso_mensual|fecha_registro|tipo_cliente|           email|
+----------+------+------+----+---------------+--------------+------------+----------------+
|         4|Carlos|  CDMX|  31|           NULL|    2026-02-10|      Básico|carlos@email.com|
+----------+------+------+----+---------------+--------------+------------+----------------+



También existe:

```python
isNotNull()
```

In [28]:
df_clientes.filter(
    F.col("email").isNotNull()
).show()

+----------+------+---------+----+---------------+--------------+------------+----------------+
|id_cliente|nombre|   estado|edad|ingreso_mensual|fecha_registro|tipo_cliente|           email|
+----------+------+---------+----+---------------+--------------+------------+----------------+
|         1|   Ana|     CDMX|  28|        18500.0|    2026-01-15|     Premium|   ana@email.com|
|         2|  Luis|  Jalisco|  35|        24000.0|    2025-11-03|      Básico|  luis@email.com|
|         4|Carlos|     CDMX|  31|           NULL|    2026-02-10|      Básico|carlos@email.com|
|         5| Sofía|   Puebla|  24|        16200.0|    2025-08-18|      Básico| sofia@email.com|
|         6| Jorge|  Jalisco|  53|        42000.0|    2023-05-06|     Premium| jorge@email.com|
|         8| Diego|Querétaro|  29|        21500.0|    2026-03-02|        NULL| diego@email.com|
|         9| Laura|   Puebla|  38|        27800.0|    2025-02-14|     Premium| laura@email.com|
|        10| Pedro|     CDMX|  61|      

### 11.2 Contar valores nulos por columna

Una forma útil de inspeccionar la calidad de los datos es contar los nulos de cada variable.


In [29]:
df_clientes.select(
    [
        F.sum(
            F.col(c).isNull().cast("int")
        ).alias(c)
        for c in df_clientes.columns
    ]
).show()

+----------+------+------+----+---------------+--------------+------------+-----+
|id_cliente|nombre|estado|edad|ingreso_mensual|fecha_registro|tipo_cliente|email|
+----------+------+------+----+---------------+--------------+------------+-----+
|         0|     0|     1|   0|              1|             1|           1|    2|
+----------+------+------+----+---------------+--------------+------------+-----+



El resultado mostrará cuántos valores faltantes existen en cada columna.

Este tipo de revisión suele formar parte de una etapa inicial de calidad de datos.

## 12. Reemplazar valores nulos con `fillna()`

Podemos reemplazar valores faltantes utilizando:

```python
fillna()
```

Por ejemplo:

In [30]:
df_fill = df_clientes.fillna({
    "estado": "Desconocido",
    "tipo_cliente": "Sin clasificar",
    "email": "Sin email"
})

df_fill.show(truncate=False)

+----------+------+-----------+----+---------------+--------------+--------------+----------------+
|id_cliente|nombre|estado     |edad|ingreso_mensual|fecha_registro|tipo_cliente  |email           |
+----------+------+-----------+----+---------------+--------------+--------------+----------------+
|1         |Ana   |CDMX       |28  |18500.0        |2026-01-15    |Premium       |ana@email.com   |
|2         |Luis  |Jalisco    |35  |24000.0        |2025-11-03    |Básico        |luis@email.com  |
|3         |Marta |Nuevo León |42  |31500.0        |2024-07-21    |Premium       |Sin email       |
|4         |Carlos|CDMX       |31  |NULL           |2026-02-10    |Básico        |carlos@email.com|
|5         |Sofía |Puebla     |24  |16200.0        |2025-08-18    |Básico        |sofia@email.com |
|6         |Jorge |Jalisco    |53  |42000.0        |2023-05-06    |Premium       |jorge@email.com |
|7         |Elena |CDMX       |47  |37000.0        |2024-12-11    |Premium       |Sin email       |


Podemos especificar diferentes valores de reemplazo dependiendo de la columna.

### 12.1 Imputar una variable numérica

Para una variable como `ingreso_mensual` podemos utilizar, por ejemplo, el promedio.

Primero calculamos:

In [31]:
ingreso_promedio = (
    df_clientes
    .select(
        F.avg("ingreso_mensual").alias("promedio")
    )
    .first()["promedio"]
)

print(f"Ingreso promedio: {ingreso_promedio:.2f}")

Ingreso promedio: 29290.91


Posteriormente:

In [32]:
df_imputado = (
    df_clientes
    .fillna({
        "ingreso_mensual": ingreso_promedio
    })
)

df_imputado.select(
    "nombre",
    "ingreso_mensual"
).show()

+------+------------------+
|nombre|   ingreso_mensual|
+------+------------------+
|   Ana|           18500.0|
|  Luis|           24000.0|
| Marta|           31500.0|
|Carlos|29290.909090909092|
| Sofía|           16200.0|
| Jorge|           42000.0|
| Elena|           37000.0|
| Diego|           21500.0|
| Laura|           27800.0|
| Pedro|           46500.0|
|Andrea|           23200.0|
|Miguel|           34000.0|
+------+------------------+



Este ejemplo tiene únicamente fines prácticos.

En un proyecto real, el método de imputación debe decidirse considerando el significado de la variable y el objetivo del análisis.

## 13. Eliminar registros con valores nulos

También podemos eliminar filas mediante:

```python
dropna()
```

Por ejemplo, eliminar registros donde falte cualquier dato:

In [33]:
df_clientes.dropna().show()

+----------+------+-------+----+---------------+--------------+------------+---------------+
|id_cliente|nombre| estado|edad|ingreso_mensual|fecha_registro|tipo_cliente|          email|
+----------+------+-------+----+---------------+--------------+------------+---------------+
|         1|   Ana|   CDMX|  28|        18500.0|    2026-01-15|     Premium|  ana@email.com|
|         2|  Luis|Jalisco|  35|        24000.0|    2025-11-03|      Básico| luis@email.com|
|         5| Sofía| Puebla|  24|        16200.0|    2025-08-18|      Básico|sofia@email.com|
|         6| Jorge|Jalisco|  53|        42000.0|    2023-05-06|     Premium|jorge@email.com|
|         9| Laura| Puebla|  38|        27800.0|    2025-02-14|     Premium|laura@email.com|
|        10| Pedro|   CDMX|  61|        46500.0|    2022-09-30|     Premium|pedro@email.com|
+----------+------+-------+----+---------------+--------------+------------+---------------+



Esto suele ser una medida bastante agresiva.

Una opción más controlada consiste en especificar columnas:

In [34]:
df_clientes.dropna(
    subset=[
        "estado",
        "ingreso_mensual"
    ]
).show()

+----------+------+----------+----+---------------+--------------+------------+----------------+
|id_cliente|nombre|    estado|edad|ingreso_mensual|fecha_registro|tipo_cliente|           email|
+----------+------+----------+----+---------------+--------------+------------+----------------+
|         1|   Ana|      CDMX|  28|        18500.0|    2026-01-15|     Premium|   ana@email.com|
|         2|  Luis|   Jalisco|  35|        24000.0|    2025-11-03|      Básico|  luis@email.com|
|         3| Marta|Nuevo León|  42|        31500.0|    2024-07-21|     Premium|            NULL|
|         5| Sofía|    Puebla|  24|        16200.0|    2025-08-18|      Básico| sofia@email.com|
|         6| Jorge|   Jalisco|  53|        42000.0|    2023-05-06|     Premium| jorge@email.com|
|         7| Elena|      CDMX|  47|        37000.0|    2024-12-11|     Premium|            NULL|
|         8| Diego| Querétaro|  29|        21500.0|    2026-03-02|        NULL| diego@email.com|
|         9| Laura|    Puebla|

Así solamente eliminamos registros donde alguna de esas variables sea nula.

## 14. Funciones para trabajar con texto

PySpark incluye numerosas funciones para limpiar y transformar columnas de texto.

Utilizaremos algunas de las más comunes.

### 14.1 Convertir a mayúsculas y minúsculas

In [35]:
df_clientes.select(
    "nombre",
    F.upper("nombre").alias("nombre_mayusculas"),
    F.lower("nombre").alias("nombre_minusculas")
).show()

+------+-----------------+-----------------+
|nombre|nombre_mayusculas|nombre_minusculas|
+------+-----------------+-----------------+
|   Ana|              ANA|              ana|
|  Luis|             LUIS|             luis|
| Marta|            MARTA|            marta|
|Carlos|           CARLOS|           carlos|
| Sofía|            SOFÍA|            sofía|
| Jorge|            JORGE|            jorge|
| Elena|            ELENA|            elena|
| Diego|            DIEGO|            diego|
| Laura|            LAURA|            laura|
| Pedro|            PEDRO|            pedro|
|Andrea|           ANDREA|           andrea|
|Miguel|           MIGUEL|           miguel|
+------+-----------------+-----------------+



### 14.2 Longitud de texto

In [36]:
df_clientes.select(
    "nombre",
    F.length("nombre").alias("longitud_nombre")
).show()

+------+---------------+
|nombre|longitud_nombre|
+------+---------------+
|   Ana|              3|
|  Luis|              4|
| Marta|              5|
|Carlos|              6|
| Sofía|              5|
| Jorge|              5|
| Elena|              5|
| Diego|              5|
| Laura|              5|
| Pedro|              5|
|Andrea|              6|
|Miguel|              6|
+------+---------------+



### 14.3 Concatenar columnas

Podemos construir una descripción utilizando `concat()`:

In [37]:
df_clientes.select(
    "nombre",
    "estado",
    F.concat(
        F.col("nombre"),
        F.lit(" - "),
        F.coalesce(
            F.col("estado"),
            F.lit("Sin estado")
        )
    ).alias("cliente_estado")
).show(truncate=False)

+------+----------+-------------------+
|nombre|estado    |cliente_estado     |
+------+----------+-------------------+
|Ana   |CDMX      |Ana - CDMX         |
|Luis  |Jalisco   |Luis - Jalisco     |
|Marta |Nuevo León|Marta - Nuevo León |
|Carlos|CDMX      |Carlos - CDMX      |
|Sofía |Puebla    |Sofía - Puebla     |
|Jorge |Jalisco   |Jorge - Jalisco    |
|Elena |CDMX      |Elena - CDMX       |
|Diego |Querétaro |Diego - Querétaro  |
|Laura |Puebla    |Laura - Puebla     |
|Pedro |CDMX      |Pedro - CDMX       |
|Andrea|Jalisco   |Andrea - Jalisco   |
|Miguel|NULL      |Miguel - Sin estado|
+------+----------+-------------------+



Aquí utilizamos también:

```python
F.coalesce()
```

para tomar el primer valor no nulo entre varias expresiones.

No debemos confundir esta función SQL con:

```python
df.coalesce()
```

que utilizamos anteriormente para reducir particiones.


## 15. Buscar patrones de texto

Podemos utilizar `contains()`:

In [38]:
df_clientes.filter(
    F.col("email").contains("@email.com")
).show()

+----------+------+---------+----+---------------+--------------+------------+----------------+
|id_cliente|nombre|   estado|edad|ingreso_mensual|fecha_registro|tipo_cliente|           email|
+----------+------+---------+----+---------------+--------------+------------+----------------+
|         1|   Ana|     CDMX|  28|        18500.0|    2026-01-15|     Premium|   ana@email.com|
|         2|  Luis|  Jalisco|  35|        24000.0|    2025-11-03|      Básico|  luis@email.com|
|         4|Carlos|     CDMX|  31|           NULL|    2026-02-10|      Básico|carlos@email.com|
|         5| Sofía|   Puebla|  24|        16200.0|    2025-08-18|      Básico| sofia@email.com|
|         6| Jorge|  Jalisco|  53|        42000.0|    2023-05-06|     Premium| jorge@email.com|
|         8| Diego|Querétaro|  29|        21500.0|    2026-03-02|        NULL| diego@email.com|
|         9| Laura|   Puebla|  38|        27800.0|    2025-02-14|     Premium| laura@email.com|
|        10| Pedro|     CDMX|  61|      

También podemos buscar prefijos:

In [39]:
df_clientes.filter(
    F.col("nombre").startswith("A")
).show()

+----------+------+-------+----+---------------+--------------+------------+----------------+
|id_cliente|nombre| estado|edad|ingreso_mensual|fecha_registro|tipo_cliente|           email|
+----------+------+-------+----+---------------+--------------+------------+----------------+
|         1|   Ana|   CDMX|  28|        18500.0|    2026-01-15|     Premium|   ana@email.com|
|        11|Andrea|Jalisco|  33|        23200.0|          NULL|      Básico|andrea@email.com|
+----------+------+-------+----+---------------+--------------+------------+----------------+



o sufijos:

In [40]:
df_clientes.filter(
    F.col("email").endswith(".com")
).show()

+----------+------+---------+----+---------------+--------------+------------+----------------+
|id_cliente|nombre|   estado|edad|ingreso_mensual|fecha_registro|tipo_cliente|           email|
+----------+------+---------+----+---------------+--------------+------------+----------------+
|         1|   Ana|     CDMX|  28|        18500.0|    2026-01-15|     Premium|   ana@email.com|
|         2|  Luis|  Jalisco|  35|        24000.0|    2025-11-03|      Básico|  luis@email.com|
|         4|Carlos|     CDMX|  31|           NULL|    2026-02-10|      Básico|carlos@email.com|
|         5| Sofía|   Puebla|  24|        16200.0|    2025-08-18|      Básico| sofia@email.com|
|         6| Jorge|  Jalisco|  53|        42000.0|    2023-05-06|     Premium| jorge@email.com|
|         8| Diego|Querétaro|  29|        21500.0|    2026-03-02|        NULL| diego@email.com|
|         9| Laura|   Puebla|  38|        27800.0|    2025-02-14|     Premium| laura@email.com|
|        10| Pedro|     CDMX|  61|      

### 15.1 Expresiones regulares

Spark también soporta expresiones regulares mediante:

```python
regexp_replace()
```

Por ejemplo, eliminaremos `@email.com` de los correos:

In [41]:
df_clientes.select(
    "email",
    F.regexp_replace(
        F.col("email"),
        "@email.com",
        ""
    ).alias("usuario_email")
).show()

+----------------+-------------+
|           email|usuario_email|
+----------------+-------------+
|   ana@email.com|          ana|
|  luis@email.com|         luis|
|            NULL|         NULL|
|carlos@email.com|       carlos|
| sofia@email.com|        sofia|
| jorge@email.com|        jorge|
|            NULL|         NULL|
| diego@email.com|        diego|
| laura@email.com|        laura|
| pedro@email.com|        pedro|
|andrea@email.com|       andrea|
|miguel@email.com|       miguel|
+----------------+-------------+



Las expresiones regulares serán especialmente útiles en procesos de limpieza de texto.

## Trabajar correctamente con fechas

Actualmente nuestra columna:

```text
fecha_registro
```

está almacenada como:

```text
string
```

Podemos comprobarlo:


In [42]:
df_clientes.printSchema()

root
 |-- id_cliente: integer (nullable = false)
 |-- nombre: string (nullable = false)
 |-- estado: string (nullable = true)
 |-- edad: integer (nullable = true)
 |-- ingreso_mensual: double (nullable = true)
 |-- fecha_registro: string (nullable = true)
 |-- tipo_cliente: string (nullable = true)
 |-- email: string (nullable = true)



Trabajar con fechas como texto limita las operaciones que Spark puede realizar correctamente.

Por ello, debemos convertirla a un tipo de fecha.

### 16.1 Convertir string a `date`

Utilizaremos:

```python
F.to_date()
```

In [43]:
df_fechas = (
    df_clientes
    .withColumn(
        "fecha_registro",
        F.to_date(
            F.col("fecha_registro"),
            "yyyy-MM-dd"
        )
    )
)

df_fechas.printSchema()

root
 |-- id_cliente: integer (nullable = false)
 |-- nombre: string (nullable = false)
 |-- estado: string (nullable = true)
 |-- edad: integer (nullable = true)
 |-- ingreso_mensual: double (nullable = true)
 |-- fecha_registro: date (nullable = true)
 |-- tipo_cliente: string (nullable = true)
 |-- email: string (nullable = true)



### 16.2 Extraer componentes de una fecha

Una vez que la columna tiene el tipo adecuado podemos obtener:

In [44]:
df_fechas.select(
    "nombre",
    "fecha_registro",
    F.year("fecha_registro").alias("anio"),
    F.month("fecha_registro").alias("mes"),
    F.dayofmonth("fecha_registro").alias("dia")
).show()

+------+--------------+----+----+----+
|nombre|fecha_registro|anio| mes| dia|
+------+--------------+----+----+----+
|   Ana|    2026-01-15|2026|   1|  15|
|  Luis|    2025-11-03|2025|  11|   3|
| Marta|    2024-07-21|2024|   7|  21|
|Carlos|    2026-02-10|2026|   2|  10|
| Sofía|    2025-08-18|2025|   8|  18|
| Jorge|    2023-05-06|2023|   5|   6|
| Elena|    2024-12-11|2024|  12|  11|
| Diego|    2026-03-02|2026|   3|   2|
| Laura|    2025-02-14|2025|   2|  14|
| Pedro|    2022-09-30|2022|   9|  30|
|Andrea|          NULL|NULL|NULL|NULL|
|Miguel|    2024-04-19|2024|   4|  19|
+------+--------------+----+----+----+



### 16.3 Calcular antigüedad

Podemos calcular la cantidad aproximada de días desde el registro hasta una fecha de referencia.

Para mantener el ejercicio reproducible utilizaremos una fecha fija:

In [45]:
fecha_referencia = F.to_date(
    F.lit("2026-09-27")
)

Después:

In [46]:
df_fechas.select(
    "nombre",
    "fecha_registro",
    F.datediff(
        fecha_referencia,
        F.col("fecha_registro")
    ).alias("dias_desde_registro")
).show()

+------+--------------+-------------------+
|nombre|fecha_registro|dias_desde_registro|
+------+--------------+-------------------+
|   Ana|    2026-01-15|                255|
|  Luis|    2025-11-03|                328|
| Marta|    2024-07-21|                798|
|Carlos|    2026-02-10|                229|
| Sofía|    2025-08-18|                405|
| Jorge|    2023-05-06|               1240|
| Elena|    2024-12-11|                655|
| Diego|    2026-03-02|                209|
| Laura|    2025-02-14|                590|
| Pedro|    2022-09-30|               1458|
|Andrea|          NULL|               NULL|
|Miguel|    2024-04-19|                891|
+------+--------------+-------------------+



Utilizar una fecha fija evita que el resultado cambie dependiendo del día en que volvamos a ejecutar el notebook.

## 17. Ordenar registros

Podemos ordenar los datos utilizando:

```python
orderBy()
```

Por ejemplo:

In [47]:
df_clientes.orderBy(
    "edad"
).show()

+----------+------+----------+----+---------------+--------------+------------+----------------+
|id_cliente|nombre|    estado|edad|ingreso_mensual|fecha_registro|tipo_cliente|           email|
+----------+------+----------+----+---------------+--------------+------------+----------------+
|         5| Sofía|    Puebla|  24|        16200.0|    2025-08-18|      Básico| sofia@email.com|
|         1|   Ana|      CDMX|  28|        18500.0|    2026-01-15|     Premium|   ana@email.com|
|         8| Diego| Querétaro|  29|        21500.0|    2026-03-02|        NULL| diego@email.com|
|         4|Carlos|      CDMX|  31|           NULL|    2026-02-10|      Básico|carlos@email.com|
|        11|Andrea|   Jalisco|  33|        23200.0|          NULL|      Básico|andrea@email.com|
|         2|  Luis|   Jalisco|  35|        24000.0|    2025-11-03|      Básico|  luis@email.com|
|         9| Laura|    Puebla|  38|        27800.0|    2025-02-14|     Premium| laura@email.com|
|         3| Marta|Nuevo León|

Orden descendente:

In [48]:
df_clientes.orderBy(
    F.col("ingreso_mensual").desc()
).show()

+----------+------+----------+----+---------------+--------------+------------+----------------+
|id_cliente|nombre|    estado|edad|ingreso_mensual|fecha_registro|tipo_cliente|           email|
+----------+------+----------+----+---------------+--------------+------------+----------------+
|        10| Pedro|      CDMX|  61|        46500.0|    2022-09-30|     Premium| pedro@email.com|
|         6| Jorge|   Jalisco|  53|        42000.0|    2023-05-06|     Premium| jorge@email.com|
|         7| Elena|      CDMX|  47|        37000.0|    2024-12-11|     Premium|            NULL|
|        12|Miguel|      NULL|  45|        34000.0|    2024-04-19|      Básico|miguel@email.com|
|         3| Marta|Nuevo León|  42|        31500.0|    2024-07-21|     Premium|            NULL|
|         9| Laura|    Puebla|  38|        27800.0|    2025-02-14|     Premium| laura@email.com|
|         2|  Luis|   Jalisco|  35|        24000.0|    2025-11-03|      Básico|  luis@email.com|
|        11|Andrea|   Jalisco|

### 17.1 Ordenar por múltiples columnas

In [49]:
df_clientes.orderBy(
    F.col("estado").asc(),
    F.col("ingreso_mensual").desc()
).show()

+----------+------+----------+----+---------------+--------------+------------+----------------+
|id_cliente|nombre|    estado|edad|ingreso_mensual|fecha_registro|tipo_cliente|           email|
+----------+------+----------+----+---------------+--------------+------------+----------------+
|        12|Miguel|      NULL|  45|        34000.0|    2024-04-19|      Básico|miguel@email.com|
|        10| Pedro|      CDMX|  61|        46500.0|    2022-09-30|     Premium| pedro@email.com|
|         7| Elena|      CDMX|  47|        37000.0|    2024-12-11|     Premium|            NULL|
|         1|   Ana|      CDMX|  28|        18500.0|    2026-01-15|     Premium|   ana@email.com|
|         4|Carlos|      CDMX|  31|           NULL|    2026-02-10|      Básico|carlos@email.com|
|         6| Jorge|   Jalisco|  53|        42000.0|    2023-05-06|     Premium| jorge@email.com|
|         2|  Luis|   Jalisco|  35|        24000.0|    2025-11-03|      Básico|  luis@email.com|
|        11|Andrea|   Jalisco|

Esto permite definir distintos criterios de ordenamiento.

## 18. Eliminar columnas

Podemos eliminar columnas mediante:

```python
drop()
```

Por ejemplo:

In [50]:
df_sin_email = df_clientes.drop("email")

df_sin_email.show()

+----------+------+----------+----+---------------+--------------+------------+
|id_cliente|nombre|    estado|edad|ingreso_mensual|fecha_registro|tipo_cliente|
+----------+------+----------+----+---------------+--------------+------------+
|         1|   Ana|      CDMX|  28|        18500.0|    2026-01-15|     Premium|
|         2|  Luis|   Jalisco|  35|        24000.0|    2025-11-03|      Básico|
|         3| Marta|Nuevo León|  42|        31500.0|    2024-07-21|     Premium|
|         4|Carlos|      CDMX|  31|           NULL|    2026-02-10|      Básico|
|         5| Sofía|    Puebla|  24|        16200.0|    2025-08-18|      Básico|
|         6| Jorge|   Jalisco|  53|        42000.0|    2023-05-06|     Premium|
|         7| Elena|      CDMX|  47|        37000.0|    2024-12-11|     Premium|
|         8| Diego| Querétaro|  29|        21500.0|    2026-03-02|        NULL|
|         9| Laura|    Puebla|  38|        27800.0|    2025-02-14|     Premium|
|        10| Pedro|      CDMX|  61|     

Podemos eliminar varias:

In [51]:
df_reducido = df_clientes.drop(
    "email",
    "tipo_cliente"
)

## Registros duplicados

Podemos utilizar:

```python
distinct()
```

para obtener filas únicas.

In [52]:
df_clientes.distinct().show()

+----------+------+----------+----+---------------+--------------+------------+----------------+
|id_cliente|nombre|    estado|edad|ingreso_mensual|fecha_registro|tipo_cliente|           email|
+----------+------+----------+----+---------------+--------------+------------+----------------+
|         1|   Ana|      CDMX|  28|        18500.0|    2026-01-15|     Premium|   ana@email.com|
|         2|  Luis|   Jalisco|  35|        24000.0|    2025-11-03|      Básico|  luis@email.com|
|         5| Sofía|    Puebla|  24|        16200.0|    2025-08-18|      Básico| sofia@email.com|
|         6| Jorge|   Jalisco|  53|        42000.0|    2023-05-06|     Premium| jorge@email.com|
|         9| Laura|    Puebla|  38|        27800.0|    2025-02-14|     Premium| laura@email.com|
|        10| Pedro|      CDMX|  61|        46500.0|    2022-09-30|     Premium| pedro@email.com|
|         3| Marta|Nuevo León|  42|        31500.0|    2024-07-21|     Premium|            NULL|
|         4|Carlos|      CDMX|

Como nuestro dataset original no contiene duplicados completos, vamos a crear un pequeño ejemplo.

In [53]:
df_duplicados = df_clientes.union(
    df_clientes.filter(
        F.col("id_cliente").isin(1, 2)
    )
)

print(
    "Registros antes:",
    df_duplicados.count()
)

Registros antes: 14


Eliminamos duplicados:

In [54]:
df_sin_duplicados = df_duplicados.distinct()

print(
    "Registros después:",
    df_sin_duplicados.count()
)

Registros después: 12


### 19.1 `dropDuplicates()`

También podemos eliminar duplicados según columnas específicas.

In [55]:
df_duplicados.dropDuplicates(
    ["id_cliente"]
).show()

+----------+------+----------+----+---------------+--------------+------------+----------------+
|id_cliente|nombre|    estado|edad|ingreso_mensual|fecha_registro|tipo_cliente|           email|
+----------+------+----------+----+---------------+--------------+------------+----------------+
|         1|   Ana|      CDMX|  28|        18500.0|    2026-01-15|     Premium|   ana@email.com|
|         2|  Luis|   Jalisco|  35|        24000.0|    2025-11-03|      Básico|  luis@email.com|
|         3| Marta|Nuevo León|  42|        31500.0|    2024-07-21|     Premium|            NULL|
|         4|Carlos|      CDMX|  31|           NULL|    2026-02-10|      Básico|carlos@email.com|
|         5| Sofía|    Puebla|  24|        16200.0|    2025-08-18|      Básico| sofia@email.com|
|         6| Jorge|   Jalisco|  53|        42000.0|    2023-05-06|     Premium| jorge@email.com|
|         7| Elena|      CDMX|  47|        37000.0|    2024-12-11|     Premium|            NULL|
|         8| Diego| Querétaro|

Esto resulta útil cuando tenemos un identificador que debería ser único.

## 20. Pipeline integrado de limpieza

Ahora combinaremos varias de las operaciones anteriores en un único pipeline.

Partiremos de:

```python
df_clientes
```

y realizaremos las siguientes transformaciones:

1. convertir la fecha a tipo `date`;
2. reemplazar categorías faltantes;
3. imputar el ingreso faltante;
4. crear ingreso anual;
5. crear grupo de edad;
6. crear antigüedad;
7. normalizar el nombre;
8. conservar únicamente las columnas necesarias.

In [57]:
df_limpio = (
    df_clientes

    .fillna({
        "estado": "Desconocido",
        "tipo_cliente": "Sin clasificar",
        "ingreso_mensual": ingreso_promedio
    })

    .withColumn(
        "fecha_registro",
        F.to_date(
            F.col("fecha_registro"),
            "yyyy-MM-dd"
        )
    )

    .withColumn(
        "nombre",
        F.initcap(
            F.trim(
                F.col("nombre")
            )
        )
    )

    .withColumn(
        "ingreso_anual",
        F.round(
            F.col("ingreso_mensual") * 12,
            2
        )
    )

    .withColumn(
        "grupo_edad",
        F.when(
            F.col("edad") < 30,
            "Menor de 30"
        )
        .when(
            F.col("edad") < 40,
            "30-39"
        )
        .when(
            F.col("edad") < 50,
            "40-49"
        )
        .otherwise("50 o más")
    )

    .withColumn(
        "dias_desde_registro",
        F.datediff(
            F.to_date(F.lit("2026-09-27")),
            F.col("fecha_registro")
        )
    )

    .select(
        "id_cliente",
        "nombre",
        "estado",
        "edad",
        "grupo_edad",
        "ingreso_mensual",
        "ingreso_anual",
        "tipo_cliente",
        "fecha_registro",
        "dias_desde_registro"
    )
)

Visualizamos el resultado:

In [58]:
df_limpio.show(
    truncate=False
)

+----------+------+-----------+----+-----------+------------------+-------------+--------------+--------------+-------------------+
|id_cliente|nombre|estado     |edad|grupo_edad |ingreso_mensual   |ingreso_anual|tipo_cliente  |fecha_registro|dias_desde_registro|
+----------+------+-----------+----+-----------+------------------+-------------+--------------+--------------+-------------------+
|1         |Ana   |CDMX       |28  |Menor de 30|18500.0           |222000.0     |Premium       |2026-01-15    |255                |
|2         |Luis  |Jalisco    |35  |30-39      |24000.0           |288000.0     |Básico        |2025-11-03    |328                |
|3         |Marta |Nuevo León |42  |40-49      |31500.0           |378000.0     |Premium       |2024-07-21    |798                |
|4         |Carlos|CDMX       |31  |30-39      |29290.909090909092|351490.91    |Básico        |2026-02-10    |229                |
|5         |Sofía |Puebla     |24  |Menor de 30|16200.0           |194400.0 

Y revisamos el schema:

In [59]:
df_limpio.printSchema()

root
 |-- id_cliente: integer (nullable = false)
 |-- nombre: string (nullable = false)
 |-- estado: string (nullable = false)
 |-- edad: integer (nullable = true)
 |-- grupo_edad: string (nullable = false)
 |-- ingreso_mensual: double (nullable = false)
 |-- ingreso_anual: double (nullable = true)
 |-- tipo_cliente: string (nullable = false)
 |-- fecha_registro: date (nullable = true)
 |-- dias_desde_registro: integer (nullable = true)



Este patrón se parece mucho más a un pipeline real de preparación de datos.

## 21. Inspeccionar el plan de transformación

Como ya conocemos `explain()`, podemos inspeccionar cómo Spark organizó todas estas operaciones:

In [60]:
df_limpio.explain("formatted")

== Physical Plan ==
* Project (3)
+- * Project (2)
   +- * Scan ExistingRDD (1)


(1) Scan ExistingRDD [codegen id : 1]
Output [8]: [id_cliente#0, nombre#1, estado#2, edad#3, ingreso_mensual#4, fecha_registro#5, tipo_cliente#6, email#7]
Arguments: [id_cliente#0, nombre#1, estado#2, edad#3, ingreso_mensual#4, fecha_registro#5, tipo_cliente#6, email#7], MapPartitionsRDD[4] at applySchemaToPythonRDD at NativeMethodAccessorImpl.java:0, ExistingRDD, UnknownPartitioning(0)

(2) Project [codegen id : 1]
Output [7]: [id_cliente#0, initcap(trim(nombre#1, None)) AS nombre#925, coalesce(estado#2, Desconocido) AS estado#921, edad#3, coalesce(nanvl(ingreso_mensual#4, null), 29290.909090909092) AS ingreso_mensual#922, cast(gettimestamp(fecha_registro#5, yyyy-MM-dd, TimestampType, try_to_date, Some(America/Mexico_City), true) as date) AS fecha_registro#924, coalesce(tipo_cliente#6, Sin clasificar) AS tipo_cliente#923]
Input [8]: [id_cliente#0, nombre#1, estado#2, edad#3, ingreso_mensual#4, fecha_regi

Aunque escribimos numerosos:

```python
withColumn()
```

Spark puede combinar varias expresiones dentro de sus operadores físicos.

Esto vuelve a mostrar una idea importante del notebook anterior:

> El código PySpark que escribimos no representa necesariamente una ejecución independiente por cada línea.

## 22. Ejercicio integrador

A partir de `df_clientes`, construye un DataFrame llamado:

```python
df_clientes_analisis
```

que cumpla las siguientes condiciones.

### Transformaciones

1. Reemplazar `estado` nulo por `"Sin información"`.
2. Convertir `fecha_registro` a tipo fecha.
3. Mantener únicamente clientes de 30 años o más.
4. Crear `ingreso_anual`.
5. Crear una columna `segmento_ingreso` con:
   - menor de 20,000 → `"Bajo"`
   - entre 20,000 y 29,999 → `"Medio"`
   - 30,000 o más → `"Alto"`
   - nulo → `"Sin información"`
6. Crear una columna `email_disponible`:
   - email no nulo → `"Sí"`
   - email nulo → `"No"`
7. Conservar las columnas:

```text
id_cliente
nombre
estado
edad
ingreso_mensual
ingreso_anual
segmento_ingreso
tipo_cliente
email_disponible
fecha_registro
```

8. Ordenar el resultado por ingreso anual de mayor a menor.


### 22.1 Solución

In [61]:
df_clientes_analisis = (
    df_clientes

    .fillna({
        "estado": "Sin información"
    })

    .withColumn(
        "fecha_registro",
        F.to_date(
            F.col("fecha_registro"),
            "yyyy-MM-dd"
        )
    )

    .filter(
        F.col("edad") >= 30
    )

    .withColumn(
        "ingreso_anual",
        F.col("ingreso_mensual") * 12
    )

    .withColumn(
        "segmento_ingreso",
        F.when(
            F.col("ingreso_mensual").isNull(),
            "Sin información"
        )
        .when(
            F.col("ingreso_mensual") < 20000,
            "Bajo"
        )
        .when(
            F.col("ingreso_mensual") < 30000,
            "Medio"
        )
        .otherwise("Alto")
    )

    .withColumn(
        "email_disponible",
        F.when(
            F.col("email").isNotNull(),
            "Sí"
        )
        .otherwise("No")
    )

    .select(
        "id_cliente",
        "nombre",
        "estado",
        "edad",
        "ingreso_mensual",
        "ingreso_anual",
        "segmento_ingreso",
        "tipo_cliente",
        "email_disponible",
        "fecha_registro"
    )

    .orderBy(
        F.col("ingreso_anual").desc_nulls_last()
    )
)



In [62]:
df_clientes_analisis.show(
    truncate=False
)

+----------+------+---------------+----+---------------+-------------+----------------+------------+----------------+--------------+
|id_cliente|nombre|estado         |edad|ingreso_mensual|ingreso_anual|segmento_ingreso|tipo_cliente|email_disponible|fecha_registro|
+----------+------+---------------+----+---------------+-------------+----------------+------------+----------------+--------------+
|10        |Pedro |CDMX           |61  |46500.0        |558000.0     |Alto            |Premium     |Sí              |2022-09-30    |
|6         |Jorge |Jalisco        |53  |42000.0        |504000.0     |Alto            |Premium     |Sí              |2023-05-06    |
|7         |Elena |CDMX           |47  |37000.0        |444000.0     |Alto            |Premium     |No              |2024-12-11    |
|12        |Miguel|Sin información|45  |34000.0        |408000.0     |Alto            |Básico      |Sí              |2024-04-19    |
|3         |Marta |Nuevo León     |42  |31500.0        |378000.0     

## 23. Conclusiones

En este notebook practicamos las operaciones fundamentales para manipular DataFrames con PySpark.

Trabajamos con:

```text
select()
filter()
where()
withColumn()
alias()
lit()
when()
cast()
fillna()
dropna()
drop()
distinct()
dropDuplicates()
orderBy()
```

También utilizamos funciones para trabajar con:

- valores nulos;
- texto;
- fechas;
- expresiones condicionales;
- columnas calculadas.

Una idea importante es que estas operaciones pueden combinarse para construir pipelines declarativos como:

```text
DataFrame
    ↓
limpieza
    ↓
filtros
    ↓
transformaciones
    ↓
nuevas variables
    ↓
selección final
    ↓
DataFrame preparado
```

Esta forma de trabajar será la base para los procesos ETL que construiremos posteriormente.

El siguiente paso será trabajar con operaciones que requieren combinar y resumir información:

- agregaciones;
- `groupBy`;
- joins;
- funciones de ventana.

Estas operaciones serán el objetivo de:

```text
02_03_aggregations_joins_windows.ipynb
```